# PG-LIF -- Collapse Diagnostic (Option B): finding the exact NaN source
**Purpose:** NOT a fix attempt. v8's membrane/plateau clamps (|vs|,|vd|<=20, p<=50) were added and FAILED --
PG-LIF collapsed again on real SHD. This notebook instruments the training loop to catch, on the real GPU
during the actual collapse, exactly which tensor goes non-finite first, so the next fix (if any) is based
on a confirmed cause instead of a third guess.

**Key insight motivating this design.** `torch.clamp(x, lo, hi)` does **not** repair NaN -- IEEE-754 defines
every comparison with NaN as false, and `clamp` is implemented via those comparisons, so NaN passes through
unchanged. It only bounds already-finite values that exceed the range. This means v8's clamps could only
ever have stopped a *magnitude* runaway (a state growing past +-20/50 while still finite) -- if the real
failure is a NaN arising some other way (e.g. an 0x(infinity) or (infinity)-(infinity) pattern inside the
surrogate gradient, or Adam's internal sqrt(variance) on a pathological gradient), the clamps are structurally
incapable of preventing it. This notebook is built to tell these two failure modes apart.

**Design, retargeted after an earlier run.** Targets plain **PGLIF** (kappa learnable, tref_p=10) at **seed 0**
-- the exact configuration and seed whose v8 (clamped) run collapsed at epoch 30. An earlier version of this
notebook targeted `PGLIF_ablE_norefrac` instead, reasoning it was the most reliable historical reproducer;
that run completed 30 clean epochs, but a different config succeeding tells us nothing about why the config
that actually failed in v8 still failed. (For the record: that ablation-E run was not wasted -- state was
pinned exactly at the clamp ceiling every single epoch, meaning the clamp was doing constant, heavy lifting
against dynamics trying to blow up continuously, even though it held.) Budget raised to 60 epochs, since
plain PGLIF's known collapse points are later (ep30 with clamps, ep37 without) than ablation E's (~ep19).
Every batch checks, in order: (1) is the **forward pass**
loss already non-finite (would mean state itself blew up, not just its gradient); (2) is the max |state|
(vs, vd, p) within safe range (should now be true thanks to the v8 clamps -- if this is ever false, the
clamps have a bug); (3) after backward(), is **every parameter's gradient** finite -- and if not, WHICH
parameter, by name, is first to go bad. On the first non-finite gradient, training stops immediately (before
`optimizer.step()` can apply garbage), a full diagnostic snapshot is saved to Drive, and the notebook prints
a clear verdict. If no collapse occurs in the epoch budget, that is also reported plainly.

In [ ]:
import os, json, time, math
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT): ROOT = '/content/drive/MyDrive'
    if not os.path.isdir(ROOT):
        raise RuntimeError("Drive mounted but no My Drive/MyDrive under /content/drive - re-run and approve.")
    BASE = os.path.join(ROOT, 'PG_LIF'); IN_COLAB = True
except ImportError:
    print('WARNING: not in Colab - local non-persistent folder.'); BASE = './PG_LIF'
except Exception as e:
    raise RuntimeError(f'Drive did not mount ({e}). Fix the mount before running so results persist.') from e
DATA = os.path.join(BASE, 'data', 'SHD')
OUT = os.path.join(BASE, 'diagnostics', 'collapse_' + time.strftime('%Y%m%d_%H%M%S'))
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print('Diagnostic output folder:', OUT)

In [ ]:
# --- CONFIG ---
T_BINS, N_IN, N_OUT, HIDDEN = 250, 700, 20, 128
BATCH, LR, MAX_TIME = 64, 5e-4, 1.4
EPOCHS = 60                 # plain PG-LIF's v8 collapse was at ep30 (this seed) and ep37 (a different seed
                             # in v7) -- 60 gives real margin past both known collapse points, not just one
DROPOUT = 0.1
SCHEDULE = [40, 80]         # will not be reached at EPOCHS=30; kept for identical optimizer behavior to v8
SEED = 0                    # matches the EXACT (config, seed) that collapsed in the v8 re-test (plain
                             # PGLIF, seed 0, epoch 30) -- retargeting to reproduce THAT specific failure,
                             # not a different config's historical collapse point (see note below)
V_CLAMP, P_CLAMP = 20.0, 50.0   # the v8 clamps, KEPT IN PLACE (diagnostic, not a rollback) so we can see
                                 # directly whether state ever approaches/hits them before the NaN, or
                                 # whether the NaN appears with state still comfortably within bounds
# Target: the single most reliable, cheapest reproducer.
TARGET_KW = dict(theta_d=1.0, P0=1.0, tau_p=T_BINS/2, tref_p=10, kappa0=1.0)   # plain PGLIF (the actual
        # config whose v8 (clamped) run collapsed at seed 0, epoch 30 -- CORRECTED from an earlier version
        # of this notebook that targeted ablation E instead, a different config whose clamped run did NOT
        # collapse in 30 epochs. That ablation-E result was real and informative (state was pinned exactly
        # at the clamp ceiling every single epoch -- the underlying dynamics are trying to blow up
        # continuously, and the clamp held), but it does not tell us what happened in the run that actually
        # failed. This version targets that run directly. NOTE ALSO: v7 (unclamped) plain PGLIF collapsed
        # on SEED 1 at epoch 37; v8 (clamped) plain PGLIF collapsed on SEED 0 at epoch 30 -- the specific
        # seed that fails is not fixed across code versions, suggesting the failure sits close to a
        # numerical knife-edge where even adding a clamp operation (which changes floating-point rounding
        # even when the clamp itself never activates) can shift which run tips over. If SEED=0 does not
        # reproduce here, try SEED=1 and SEED=2 before concluding anything -- given this sensitivity, a
        # single clean seed is weaker evidence of "no bug" than it would be for a more numerically stable
        # model.
print(f'Target: plain PGLIF (tref_p={TARGET_KW["tref_p"]}, the config that collapsed in v8), seed={SEED}, up to {EPOCHS} epochs.')

In [ ]:
import numpy as np, h5py, torch, torch.nn as nn, gzip, shutil, urllib.request
for name, url in {'shd_train.h5':'https://zenkelab.org/datasets/shd_train.h5.gz',
                  'shd_test.h5':'https://zenkelab.org/datasets/shd_test.h5.gz'}.items():
    dst = os.path.join(DATA, name)
    if not os.path.exists(dst):
        gz = dst+'.gz'; print('downloading', url); urllib.request.urlretrieve(url, gz)
        with gzip.open(gz,'rb') as fi, open(dst,'wb') as fo: shutil.copyfileobj(fi, fo)
        os.remove(gz)

def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        return ([np.array(t) for t in f['spikes']['times']],
                [np.array(u) for u in f['spikes']['units']],
                np.array(f['labels'], dtype=np.int64))
TR_raw = load_split('shd_train.h5'); TE_raw = load_split('shd_test.h5')

def precompute_dense(split):
    times, units, labels = split; n = len(labels)
    time_bins = np.linspace(0, MAX_TIME, num=T_BINS)      # official digitize/linspace binning (v6 fix)
    X = torch.zeros(n, T_BINS, N_IN, dtype=torch.bool)
    for i in range(n):
        tb = np.clip(np.digitize(times[i], time_bins), 0, T_BINS-1)
        X[i, tb, units[i]] = True
    return X, torch.as_tensor(labels, dtype=torch.long)
TR_X, TR_Y = precompute_dense(TR_raw); TE_X, TE_Y = precompute_dense(TE_raw)
print('train', len(TR_Y), 'test', len(TE_Y))

def batches(X, Y, bs, shuffle, device, drop_last=False):
    n = len(Y); idx = torch.randperm(n) if shuffle else torch.arange(n)
    if drop_last: n = (n//bs)*bs
    for b0 in range(0, n, bs):
        sel = idx[b0:b0+bs]; yield X[sel].float().to(device), Y[sel].to(device)

## Instrumented PG-LIF cell
Identical dynamics to the v8 (clamped) cell, but every forward call records, into a module-level dict, the
per-step max |vd|, |vs|, |p| seen -- cheap (a handful of `.abs().max()` calls) and lets us plot the
build-up trajectory, not just the moment of collapse.

In [ ]:
PROBE = {'vd_max': [], 'vs_max': [], 'p_max': [], 'forward_nonfinite': None}

class Triangle(torch.autograd.Function):
    gamma = 1.0
    @staticmethod
    def forward(ctx, x): ctx.save_for_backward(x); return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs()/Triangle.gamma, min=0.0)
spike_fn = Triangle.apply
def decay(tau): return math.exp(-1.0/tau)

class PGLIFCell(nn.Module):
    """Same as v8: manuscript Eqs 6-10, scaled plateau drive, WITH the V_CLAMP/P_CLAMP safeguards."""
    th = 1.0; beta = 1.0
    def __init__(self, N, theta_d=1.0, P0=1.0, tau_p=None, tref_p=10, kappa0=1.0):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200)
        ap0 = decay(tau_p or T_BINS/2)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0/(1-ap0))))
        self.kappa = nn.Parameter(torch.full((N,), float(kappa0)))
        self.P0, self.tref_p = P0, tref_p
        self.register_buffer('theta_d', torch.full((N,), float(theta_d)))
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a = z(), z(), z(), z(); self.rp = z()
    def forward(self, I_ff, I_rec):
        self.vd = self.ad*self.vd + I_ff
        self.vd = torch.clamp(self.vd, -V_CLAMP, V_CLAMP)
        PROBE['vd_max'].append(self.vd.detach().abs().max().item())
        ed = spike_fn(self.vd - self.theta_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp-1, min=0) + ed.detach()*self.tref_p
        self.p = torch.sigmoid(self.ap_logit)*self.p + self.P0*ed
        self.p = torch.clamp(self.p, max=P_CLAMP)
        PROBE['p_max'].append(self.p.detach().abs().max().item())
        self.vs = self.am*self.vs + I_ff + I_rec + self.kappa*self.p*(1-self.am)
        self.vs = torch.clamp(self.vs, -V_CLAMP, V_CLAMP)
        PROBE['vs_max'].append(self.vs.detach().abs().max().item())
        th = self.th + self.beta*self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach()*th.detach(); self.a = self.aa*self.a + s.detach()
        return s

class RecLayer(nn.Module):
    def __init__(self, n_in, n_hid, **kw):
        super().__init__()
        self.w_in = nn.Linear(n_in, n_hid); self.drop = nn.Dropout(DROPOUT)
        self.w_rec = nn.Linear(n_hid, n_hid, bias=True)   # v6 fix: default init, bias=True
        self.cell = PGLIFCell(n_hid, **kw); self.n_hid = n_hid
    def init(self, B, dev): self.cell.init(B, dev); self.s = torch.zeros(B, self.n_hid, device=dev)
    def step(self, x_t):
        iff = self.drop(self.w_in(x_t)); irec = self.w_rec(self.s)
        self.s = self.cell(iff, irec); return self.s

class RecSNN(nn.Module):
    def __init__(self, **kw):
        super().__init__()
        self.layer1 = RecLayer(N_IN, HIDDEN, **kw); self.layer2 = RecLayer(HIDDEN, HIDDEN, **kw)
        self.w_out = nn.Linear(HIDDEN, N_OUT)
    def forward(self, x):
        B, T, _ = x.shape; dev = x.device
        self.layer1.init(B, dev); self.layer2.init(B, dev)
        out = torch.zeros(B, N_OUT, device=dev)
        for t in range(T):
            out = out + self.w_out(self.layer2.step(self.layer1.step(x[:, t])))
        return out

def accuracy(model, device):
    model.eval(); c = t = 0
    with torch.no_grad():
        for x, y in batches(TE_X, TE_Y, 128, False, device):
            c += (model(x).argmax(1) == y).sum().item(); t += len(y)
    return c/t

## Instrumented training loop
Checks, in order, at every batch: (1) forward-pass loss finite? (2) after backward, is EVERY parameter's
gradient finite -- and if not, which one, by name, first? On failure: dump a full snapshot (model state,
optimizer state, the exact input batch, per-parameter grad norms, and the PROBE state-magnitude history)
to Drive, print the verdict, and stop. This is deliberately NOT wrapped in a try/except that continues --
we want the very first failure, not a later cascade.

In [ ]:
def named_grad_check(model):
    """Returns (all_finite, first_bad_name, details) without altering any gradient."""
    bad = []
    for name, p in model.named_parameters():
        if p.grad is None: continue
        finite = torch.isfinite(p.grad)
        if not finite.all():
            n_bad = (~finite).sum().item()
            bad.append((name, n_bad, p.grad.numel(), p.grad[finite].abs().max().item() if finite.any() else float('nan')))
    return (len(bad) == 0), bad

torch.manual_seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu': print('WARNING: no GPU - collapse may not reproduce faithfully; this is a real-GPU phenomenon.')
model = RecSNN(**TARGET_KW).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=SCHEDULE, gamma=0.1)
crit = nn.CrossEntropyLoss()

VERDICT = None
epoch_probe_summary = []
for ep in range(EPOCHS):
    model.train(); t0 = time.time()
    PROBE['vd_max'].clear(); PROBE['p_max'].clear(); PROBE['vs_max'].clear()
    stop = False
    for bi, (x, y) in enumerate(batches(TR_X, TR_Y, BATCH, True, device, drop_last=True)):
        opt.zero_grad()
        out = model(x)
        loss = crit(out, y)
        # CHECK 1: forward-pass loss itself
        if not torch.isfinite(loss):
            VERDICT = {'stage': 'forward_loss', 'epoch': ep, 'batch': bi,
                       'vd_max_so_far': max(PROBE['vd_max']) if PROBE['vd_max'] else None,
                       'vs_max_so_far': max(PROBE['vs_max']) if PROBE['vs_max'] else None,
                       'p_max_so_far': max(PROBE['p_max']) if PROBE['p_max'] else None}
            print(f'\n*** FORWARD PASS PRODUCED NON-FINITE LOSS at ep{ep} batch{bi} ***')
            print('State magnitudes leading up to it (this epoch, this batch\'s forward):', VERDICT)
            stop = True; break
        loss.backward()
        # CHECK 2: every parameter's gradient
        all_finite, bad = named_grad_check(model)
        if not all_finite:
            VERDICT = {'stage': 'backward_grad', 'epoch': ep, 'batch': bi, 'bad_params': bad,
                       'vd_max_this_epoch': max(PROBE['vd_max']), 'vs_max_this_epoch': max(PROBE['vs_max']),
                       'p_max_this_epoch': max(PROBE['p_max']),
                       'loss_value': loss.item()}
            print(f'\n*** NON-FINITE GRADIENT at ep{ep} batch{bi} (loss was finite: {loss.item():.3f}) ***')
            print('First bad parameter(s) [name, n_bad_elements, total_elements, max_finite_grad_elsewhere]:')
            for b in bad: print('  ', b)
            print('State magnitudes THIS epoch (should be <= clamps if clamps are working):')
            print(f"    max|vd|={VERDICT['vd_max_this_epoch']:.2f}  max|vs|={VERDICT['vs_max_this_epoch']:.2f}  max|p|={VERDICT['p_max_this_epoch']:.2f}")
            print(f"    (clamps are V_CLAMP={V_CLAMP}, P_CLAMP={P_CLAMP} -- if these maxima are well below the")
            print(f"     clamp values, the NaN did NOT come from a magnitude runaway the clamps could have caught.)")
            # save the exact offending batch + model/opt state for offline inspection
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'x': x.cpu(), 'y': y.cpu(), 'verdict': VERDICT},
                       os.path.join(OUT, f'collapse_snapshot_ep{ep}_b{bi}.pt'))
            stop = True; break
        torch.nn.utils.clip_grad_value_(model.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
    if stop: break
    sch.step()
    acc = accuracy(model, device)
    ep_summary = {'epoch': ep, 'test_acc': acc, 'vd_max': max(PROBE['vd_max']), 'vs_max': max(PROBE['vs_max']),
                  'p_max': max(PROBE['p_max']), 'time': time.time()-t0}
    epoch_probe_summary.append(ep_summary)
    print(f"ep{ep:03d}  acc {acc:.4f}  max|vd|={ep_summary['vd_max']:.2f}  max|vs|={ep_summary['vs_max']:.2f}  "
          f"max|p|={ep_summary['p_max']:.2f}  {ep_summary['time']:.0f}s")

json.dump({'verdict': VERDICT, 'epoch_summary': epoch_probe_summary,
           'config': {'SEED': SEED, 'EPOCHS': EPOCHS, 'V_CLAMP': V_CLAMP, 'P_CLAMP': P_CLAMP,
                      'target_kw': {k: str(v) for k, v in TARGET_KW.items()}}},
          open(os.path.join(OUT, 'diagnostic_result.json'), 'w'), indent=2, default=str)

## Verdict

In [ ]:
if VERDICT is None:
    print(f'NO COLLAPSE within {EPOCHS} epochs on seed {SEED} (plain PGLIF, the exact config+seed whose')
    print('v8 (clamped) run collapsed at epoch 30). Given this run used the SAME clamps and did not fail,')
    print('either: (a) this run got lucky and a later epoch or a different seed would still fail -- ')
    print('consistent with the seed-sensitivity noted above, since v7-vs-v8 already shifted which seed')
    print('failed -- or (b) this execution genuinely differs enough (GPU, library versions, float op order)')
    print('that it did not reproduce. Check the per-epoch max|vd|/max|vs|/max|p| printed above: if FAR below')
    print('the clamp values throughout, that is a real positive sign. If PINNED at the ceiling (as ablation E')
    print('was), the instability is still present and being suppressed, not resolved. Try SEED=1, SEED=2')
    print('before concluding the clamps are sufficient for this config.')
else:
    print('=== DIAGNOSTIC VERDICT ===')
    print(json.dumps(VERDICT, indent=2, default=str))
    print()
    if VERDICT['stage'] == 'forward_loss':
        print('CONCLUSION: the FORWARD pass itself produced a non-finite loss. This means some STATE variable')
        print('(or the readout accumulator `out`, which is NOT clamped) escaped to inf/nan despite the vs/vd/p')
        print('clamps. Likely culprit: `out` in RecSNN.forward accumulates w_out(spikes) over T=250 steps with')
        print('NO clamp or decay at all -- if spike rates are high for long enough, this sum can itself grow')
        print('unboundedly even with every per-step state variable safely clamped. This would be a genuinely')
        print('NEW bug location (the readout accumulator), not the plateau/a2 states v8 targeted.')
    else:
        print('CONCLUSION: the forward pass and loss were FINITE, but the BACKWARD pass produced a non-finite')
        print('gradient. Check whether the reported max|vd|/max|vs|/max|p| are well below the clamp values --')
        print('if so, this CONFIRMS the failure is not a magnitude runaway the clamps could ever have caught,')
        print('but a backward-pass numerical issue (e.g. the Triangle surrogate\'s gradient formula producing')
        print('0/0 or similar at a specific boundary, or an Adam moment-estimate issue with this gradient')
        print('pattern). The specific parameter name(s) printed above pinpoint WHERE in the model.')
    print()
    print('Snapshot saved to:', OUT)
    print('Next step: do NOT attempt a fix without discussing this verdict first -- the correct fix depends')
    print('entirely on which of the two failure modes above actually occurred.')